In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [ ]:
DATA_DIR = '.'
CITIES   = ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai']

dfs = {}
for city in CITIES:
    path = os.path.join(DATA_DIR, f'{city}.csv')
    df   = pd.read_csv(path)
    df['City'] = city
    dfs[city]  = df
    print(f'{city:12s} — {df.shape[0]:,} rows | {df.shape[1]} cols')

raw = pd.concat(dfs.values(), ignore_index=True)
print(f'\nCombined shape: {raw.shape}')

Bangalore    — 6,207 rows | 41 cols
Chennai      — 5,014 rows | 41 cols
Delhi        — 4,998 rows | 41 cols
Hyderabad    — 2,518 rows | 41 cols
Kolkata      — 6,507 rows | 41 cols
Mumbai       — 7,719 rows | 41 cols

Combined shape: (32963, 41)


In [3]:
raw.head(3)

,Price,Area,Location,No. of Bedrooms,Resale,MaintenanceStaff,Gymnasium,SwimmingPool,LandscapedGardens,JoggingTrack,RainWaterHarvesting,IndoorGames,ShoppingMall,Intercom,SportsFacility,ATM,ClubHouse,School,24X7Security,PowerBackup,CarParking,StaffQuarter,Cafeteria,MultipurposeRoom,Hospital,WashingMachine,Gasconnection,AC,Wifi,Children'splayarea,LiftAvailable,BED,VaastuCompliant,Microwave,GolfCourse,TV,DiningTable,Sofa,Wardrobe,Refrigerator,City
0,30000000,3340,JP Nagar Phase 1,4,0,1,1,1,1,1,1,1,0,1,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,Bangalore
1,7888000,1045,Dasarahalli on Tumkur Road,2,0,0,1,1,1,1,1,1,0,0,1,0,1,0,1,1,1,0,0,1,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,Bangalore
2,4866000,1179,Kannur on Thanisandra Main Road,2,0,0,1,1,1,1,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,Bangalore


In [4]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32963 entries, 0 to 32962
Data columns (total 41 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Price                32963 non-null  int64 
 1   Area                 32963 non-null  int64 
 2   Location             32963 non-null  object
 3   No. of Bedrooms      32963 non-null  int64 
 4   Resale               32963 non-null  int64 
 5   MaintenanceStaff     32963 non-null  int64 
 6   Gymnasium            32963 non-null  int64 
 7   SwimmingPool         32963 non-null  int64 
 8   LandscapedGardens    32963 non-null  int64 
 9   JoggingTrack         32963 non-null  int64 
 10  RainWaterHarvesting  32963 non-null  int64 
 11  IndoorGames          32963 non-null  int64 
 12  ShoppingMall         32963 non-null  int64 
 13  Intercom             32963 non-null  int64 
 14  SportsFacility       32963 non-null  int64 
 15  ATM                  32963 non-null  int64 
 16  Club

In [ ]:
missing     = raw.isnull().sum()
missing_pct = (missing / len(raw) * 100).round(2)
missing_df  = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df  = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print('No missing values found.')
else:
    print(missing_df)

No missing values found.


In [7]:
df = raw.copy()

# Numeric → median (robust to skew/outliers)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    if df[col].isnull().sum() > 0:
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f'  [numeric]  {col}: filled with median = {med}')

# Categorical → mode
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mod = df[col].mode()[0]
        df[col] = df[col].fillna(mod)
        print(f'  [category] {col}: filled with mode  = {mod}')

print('\nMissing values remaining:', df.isnull().sum().sum())


Missing values remaining: 0


In [8]:
n_before = len(df)
df = df.drop_duplicates()
n_after  = len(df)
print(f'Duplicates removed : {n_before - n_after}')
print(f'Rows remaining     : {n_after:,}')

Duplicates removed : 3828
Rows remaining     : 29,135


In [9]:
BINARY_COLS = [
    'Resale', 'MaintenanceStaff', 'Gymnasium', 'SwimmingPool',
    'LandscapedGardens', 'JoggingTrack', 'RainWaterHarvesting', 'IndoorGames',
    'ShoppingMall', 'Intercom', 'SportsFacility', 'ATM', 'ClubHouse', 'School',
    '24X7Security', 'PowerBackup', 'CarParking', 'StaffQuarter', 'Cafeteria',
    'MultipurposeRoom', 'Hospital', 'WashingMachine', 'Gasconnection', 'AC',
    'Wifi', "Children'splayarea", 'LiftAvailable', 'BED', 'VaastuCompliant',
    'Microwave', 'GolfCourse', 'TV', 'DiningTable', 'Sofa', 'Wardrobe', 'Refrigerator'
]
BINARY_COLS = [c for c in BINARY_COLS if c in df.columns]

for col in BINARY_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

df['Price']           = pd.to_numeric(df['Price'],           errors='coerce')
df['Area']            = pd.to_numeric(df['Area'],            errors='coerce')
df['No. of Bedrooms'] = pd.to_numeric(df['No. of Bedrooms'], errors='coerce')
df = df.dropna(subset=['Price', 'Area', 'No. of Bedrooms'])

print('Data types after fixing:')
print(df[['Price','Area','No. of Bedrooms'] + BINARY_COLS[:5]].dtypes)

Data types after fixing:
Price                int64
Area                 int64
No. of Bedrooms      int64
Resale               int64
MaintenanceStaff     int64
Gymnasium            int64
SwimmingPool         int64
LandscapedGardens    int64
dtype: object


### 7.2 IQR-Based Outlier Removal on Price & Area

In [11]:
def remove_outliers_iqr(df, col, multiplier=3.0):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR    = Q3 - Q1
    lo, hi = Q1 - multiplier * IQR, Q3 + multiplier * IQR
    mask   = df[col].between(lo, hi)
    print(f'{col:20s}: removed {(~mask).sum():,} outliers  (bounds [{lo:,.0f}, {hi:,.0f}])')
    return df[mask]

n_pre = len(df)
df    = remove_outliers_iqr(df, 'Price')
df    = remove_outliers_iqr(df, 'Area')
print(f'\nRows: {n_pre:,} → {len(df):,}')

Price               : removed 1,232 outliers  (bounds [-20,180,004, 36,660,003])
Area                : removed 478 outliers  (bounds [-1,056, 3,417])

Rows: 29,135 → 27,425


In [12]:
df = df[(df['Price'] > 0) & (df['Area'] > 0)]
df = df[df['No. of Bedrooms'].between(1, 10)]

# Binary columns must be exactly 0 or 1 (no legacy '9' values)
for col in BINARY_COLS:
    df = df[df[col].isin([0, 1])]

df = df.reset_index(drop=True)
print(f'Rows after sanity checks: {len(df):,}')

Rows after sanity checks: 7,925


In [13]:
# Price per sq ft — key derived feature
df['PricePerSqFt'] = (df['Price'] / df['Area']).round(2)

# Amenity score — total number of amenities per listing
df['AmenityScore'] = df[BINARY_COLS].sum(axis=1)

# Bedroom category — for EDA grouping
df['BedroomCategory'] = pd.cut(
    df['No. of Bedrooms'],
    bins=[0, 1, 2, 3, 10],
    labels=['1BHK', '2BHK', '3BHK', '4BHK+']
)

print('New features: PricePerSqFt, AmenityScore, BedroomCategory')
df[['Price', 'Area', 'PricePerSqFt', 'AmenityScore', 'BedroomCategory']].head()

New features: PricePerSqFt, AmenityScore, BedroomCategory


,Price,Area,PricePerSqFt,AmenityScore,BedroomCategory
0,30000000,3340,8982.04,14,4BHK+
1,7888000,1045,7548.33,15,2BHK
2,4866000,1179,4127.23,9,2BHK
3,8358000,1675,4989.85,3,3BHK
4,6845000,1670,4098.80,16,3BHK


In [ ]:
OUTPUT_PATH = 'real_estate_cleaned.csv'
df = df.reset_index(drop=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved → {OUTPUT_PATH}')
print(f'Final shape  : {df.shape}')
print(f'Columns      : {df.columns.tolist()}')

✅ Saved → real_estate_cleaned.csv
   Final shape  : (7925, 44)
   Columns      : ['Price', 'Area', 'Location', 'No. of Bedrooms', 'Resale', 'MaintenanceStaff', 'Gymnasium', 'SwimmingPool', 'LandscapedGardens', 'JoggingTrack', 'RainWaterHarvesting', 'IndoorGames', 'ShoppingMall', 'Intercom', 'SportsFacility', 'ATM', 'ClubHouse', 'School', '24X7Security', 'PowerBackup', 'CarParking', 'StaffQuarter', 'Cafeteria', 'MultipurposeRoom', 'Hospital', 'WashingMachine', 'Gasconnection', 'AC', 'Wifi', "Children'splayarea", 'LiftAvailable', 'BED', 'VaastuCompliant', 'Microwave', 'GolfCourse', 'TV', 'DiningTable', 'Sofa', 'Wardrobe', 'Refrigerator', 'City', 'PricePerSqFt', 'AmenityScore', 'BedroomCategory']
